# Coherence — RealEngine **v3**

Third iteration. Both previous versions failed their own section-7 test, in different
ways. Keeping the record, because the failures are more informative than the code.

### v1: the model copied its input

A reconstruction autoencoder with skip connections learned to pass pixels straight
through. Val loss 0.0086 looked excellent and meant the opposite of what it seemed:
the model never had to understand anything. Injecting a lesion produced *no* response
(d = +0.04). Two supporting bugs: the intensity-percentile retina mask selected vitreous
and choroid rather than retina, and the degradation model led with Gaussian blur — which
for a reconstruction task makes an image **easier** to predict, so degraded scans came
back calmer than clean ones.

### v2: inpainting hid the very thing it was meant to find

Switching to inpainting fixed the copying. Val loss rose to 0.035 on a genuinely harder
task, the fill continued the retinal layers plausibly, and the lesion no longer appeared
in the reconstruction. Degraded finally separated from clean (d = +0.49).

The anomaly signal was still exactly zero (d = +0.04). The reason, once seen, is obvious:
**the sliding hole erases the lesion before the model sees it.** The model confidently
inpaints normal retina, and all 20 passes agree — because normal retina is what it knows.
Variance goes *down* at a lesion, not up.

Inpainting variance answers "how ambiguous is the normal anatomy here". It does not
answer "is what is actually here abnormal". Different questions.

### What changed in v3

The residual — computed all along in v2 and thrown away — is the answer:

$$z = \frac{|x - \mu_{\text{fill}}|}{\sigma_{\text{fill}} + \epsilon}$$

How far the observed content sits from what the model expected, in units of how sure the
model was about that expectation.

| Pattern | Meaning | Outcome |
|---|---|---|
| high residual, low variance | the model knows what belongs here, and this isn't it | **focal → review** |
| high residual everywhere | nothing matches expectation | **diffuse → rescan** |
| low residual | as expected | **cleared** |

| | v2 | v3 |
|---|---|---|
| Score | std across passes | **residual z-score** |
| Patch / stride | 32 / 24 (visible tiling) | **16 / 12, positions batched** |
| `inject_anomaly` | smoothed the whole image (bug) | **smooths the lesion edge only** |

On a stubbed inpainter, variance-only gave d = 0.00 on every statistic; residual-z gave
+2.27 on `u_total` and +0.70 on `top_mass`. That stub had a perfect normal reference, so
treat those as an upper bound.

**Runtime:** T4 GPU. ~35 minutes end to end.

## 1. Setup

In [ ]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok python-multipart pydicom kagglehub

import os, io, glob, json, math, time, base64, uuid, random
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage
from PIL import Image
import matplotlib.pyplot as plt

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## 2. Data — normal B-scans only

Unchanged from v1. Training on normal anatomy alone is what makes anything else register
as unfamiliar. No pathology labels anywhere in this notebook.

Watch the output: if it says SYNTHETIC, the pipeline runs but no number downstream is
evidence of anything.

In [ ]:
DATA_OK, NORMAL_PATHS = False, []
try:
    import kagglehub
    root = kagglehub.dataset_download("paultimothymooney/kermany2018")
    NORMAL_PATHS = sorted(glob.glob(os.path.join(root, "**", "NORMAL", "*.jpeg"),
                                    recursive=True))
    if len(NORMAL_PATHS) > 500:
        DATA_OK = True
        print(f"Kermany loaded: {len(NORMAL_PATHS):,} normal B-scans")
except Exception as e:
    print("Kermany unavailable:", type(e).__name__, str(e)[:120])
if not DATA_OK:
    print("\n>>> SYNTHETIC FALLBACK. Pipeline runs; results are not evidence. <<<\n")

In [ ]:
H, W = 128, 256

def synth_bscan(seed):
    r = np.random.default_rng(seed)
    img = np.zeros((H, W)); x = np.arange(W)
    fx = W * r.uniform(0.40, 0.60); d = (x - fx) / (W * 0.085)
    pit = 26 * np.exp(-d * d)
    wob = sum(r.uniform(.6, 2.2) * np.sin(x * r.uniform(.004, .016) + r.uniform(0, 6.3))
              for _ in range(5))
    top = 44 + wob + pit * .95
    for i, (off, th, b) in enumerate([(0,3,.78),(3,6,.30),(9,7,.62),(16,8,.22),
                                      (24,7,.50),(31,6,.20),(37,4,.92),(41,4,1.),(45,11,.16)]):
        shrink = pit * max(0., .95 - i * .19) if i < 5 else 0.
        for k in range(th):
            yy = np.clip((top + off + k - shrink).astype(int), 0, H - 1)
            img[yy, x] = np.maximum(img[yy, x], b * (.82 + .35 * r.random(W)))
    img += r.normal(0, .09, img.shape)
    return np.clip(img, 0, 1).astype(np.float32)

def load_bscan(i):
    if DATA_OK:
        im = Image.open(NORMAL_PATHS[i % len(NORMAL_PATHS)]).convert("L").resize((W, H))
        return np.asarray(im, dtype=np.float32) / 255.0
    return synth_bscan(i)

N_TRAIN, N_VAL = (4000, 600) if DATA_OK else (1200, 300)
print(f"train={N_TRAIN} val={N_VAL} source={'Kermany' if DATA_OK else 'SYNTHETIC'}")

class NormalScans(Dataset):
    def __init__(self, lo, hi): self.lo, self.hi = lo, hi
    def __len__(self): return self.hi - self.lo
    def __getitem__(self, i): return torch.from_numpy(load_bscan(self.lo + i))[None]

train_dl = DataLoader(NormalScans(0, N_TRAIN), batch_size=32, shuffle=True,
                      num_workers=2, drop_last=True)
val_dl   = DataLoader(NormalScans(N_TRAIN, N_TRAIN + N_VAL), batch_size=32)

## 3. Model and the inpainting task

Architecture is a standard dropout U-Net — the skip connections stay, because with
inpainting they are no longer a liability. The hole is genuinely empty, so there is
nothing to copy at any scale; the skips only help carry surrounding context, which is what
you want.

`mask_patches` punches random square holes out of a batch. Loss is computed **only inside
the holes**. That single detail is the whole difference from v1: the model is never
rewarded for reproducing pixels it can see.

In [ ]:
P_DROP = 0.30
PATCH  = 16          # v2 used 32; smaller holes give a sharper, less tiled score field

def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Dropout2d(P_DROP),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Dropout2d(P_DROP))

class InpaintUNet(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.e1, self.e2, self.e3 = block(2, c), block(c, c*2), block(c*2, c*4)
        self.bott = block(c*4, c*8)
        self.u3 = nn.ConvTranspose2d(c*8, c*4, 2, 2); self.d3 = block(c*8, c*4)
        self.u2 = nn.ConvTranspose2d(c*4, c*2, 2, 2); self.d2 = block(c*4, c*2)
        self.u1 = nn.ConvTranspose2d(c*2, c,   2, 2); self.d1 = block(c*2, c)
        self.out = nn.Conv2d(c, 1, 1); self.pool = nn.MaxPool2d(2)
    def forward(self, x, holes):
        # second channel tells the model WHERE the hole is
        z = torch.cat([x, holes], 1)
        e1 = self.e1(z);             e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); b  = self.bott(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b),  e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))

def mask_patches(xb, patch=PATCH, n_holes=3):
    B, _, Hh, Ww = xb.shape
    holes = torch.zeros_like(xb)
    ys = torch.randint(0, Hh - patch, (B, n_holes))
    xs = torch.randint(0, Ww - patch, (B, n_holes))
    for b in range(B):
        for k in range(n_holes):
            holes[b, :, ys[b,k]:ys[b,k]+patch, xs[b,k]:xs[b,k]+patch] = 1.
    return xb * (1 - holes), holes

model = InpaintUNet().to(DEV)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 4. Train

Loss inside the holes only. Expect the numbers to be **much higher than v1** — predicting
hidden anatomy is a far harder task than copying visible pixels. v1's val loss of 0.0086
was a warning sign, not an achievement: it meant the model had learned to pass its input
through.

In [ ]:
EPOCHS = 18 if DATA_OK else 12
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)

def hole_loss(pred, target, holes):
    return (F.l1_loss(pred, target, reduction="none") * holes).sum() / holes.sum()

for ep in range(EPOCHS):
    model.train(); tr = 0.
    for xb in train_dl:
        xb = xb.to(DEV)
        xin, holes = mask_patches(xb)
        opt.zero_grad()
        loss = hole_loss(model(xin, holes), xb, holes)
        loss.backward(); opt.step(); tr += loss.item()
    sched.step()

    model.eval(); va = 0.
    with torch.no_grad():
        for xb in val_dl:
            xb = xb.to(DEV)
            xin, holes = mask_patches(xb)
            va += hole_loss(model(xin, holes), xb, holes).item()
    print(f"epoch {ep+1:2d}/{EPOCHS}  train {tr/len(train_dl):.4f}  val {va/len(val_dl):.4f}")

torch.save(model.state_dict(), "coherence_v3.pt")
print("saved coherence_v3.pt")

### Sanity check — did it learn anatomy, or memorise pixels?

Punch a hole in a scan and look at the fill. If the model has learned the structure, the
retinal layers should continue plausibly across the gap. If it fills with grey mush or
copies noise, the training has not worked and nothing downstream will mean anything.

**This is the check v1 needed and did not have.**

In [ ]:
model.eval()
x = load_bscan(N_TRAIN + 2)
xb = torch.from_numpy(x)[None, None].to(DEV)
holes = torch.zeros_like(xb); holes[..., 48:80, 96:128] = 1.   # 32px hole, just for the visual check
with torch.no_grad():
    fill = model(xb * (1 - holes), holes)
blend = (xb * (1 - holes) + fill * holes)[0,0].cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(13, 2.8))
for a, (im, t) in zip(ax, [(x, "original"),
                           ((xb*(1-holes))[0,0].cpu().numpy(), "hole punched"),
                           (blend, "model's fill")]):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t, fontsize=9); a.axis("off")
plt.show()

## 5. The score — sliding-window inpainting, then the residual

Slide a hole across the scan. At each position run 20 stochastic passes (dropout on), and
keep **both** the mean fill and the standard deviation across passes.

The mean fill is the model's expectation: what it believes belongs there, having seen only
the surroundings. The std is how sure it is of that expectation. v2 used the std alone and
found nothing, because the hole hides the lesion. v3 compares the expectation against what
is **actually** in the scan.

Hole positions are batched into single forward calls, so the finer stride costs far less
than 4x the time.

### The mask, rebuilt

v1's mask was the first thing that broke. This version finds the **ILM** (first sustained
bright row per A-scan) and the **RPE** (brightest row), median-filters both boundaries
across columns to enforce anatomical continuity, and rejects columns where the band is
implausibly thin. Tested on synthetic OCT with realistic sub-RPE speckle: v1 put 59% of
its mask inside the true retinal band, this puts 93%.

### Normalising by signal

OCT speckle is multiplicative — variance scales with brightness, so the RPE always looked
uncertain simply for being the brightest thing in the image. Dividing by local signal
removes that artefact.

In [ ]:
def retina_mask(x, smooth_x=15, pad_above=3, pad_below=8, min_thick=8):
    x = np.asarray(x, dtype=np.float64); Hh, Ww = x.shape
    sm = ndimage.uniform_filter(x, size=(3, smooth_x))
    rpe = np.argmax(sm, axis=0)
    thr = sm.min(axis=0) + 0.25 * (sm.max(axis=0) - sm.min(axis=0))
    sustained = ndimage.uniform_filter((sm > thr[None,:]).astype(float), size=(3,1)) > 0.99
    ilm = np.array([np.argmax(sustained[:,c]) if sustained[:,c].any() else 0
                    for c in range(Ww)])
    ilm = ndimage.median_filter(ilm, size=25); rpe = ndimage.median_filter(rpe, size=25)
    bad = (rpe - ilm) < min_thick
    mask = np.zeros((Hh, Ww), dtype=np.uint8); rows = np.arange(Hh)[:,None]
    lo = np.clip(ilm - pad_above, 0, Hh-1)[None,:]
    hi = np.clip(rpe + pad_below, 0, Hh-1)[None,:]
    mask[(rows >= lo) & (rows <= hi)] = 1; mask[:, bad] = 0
    return mask

def anomaly_score(x, mu, sd, eps=0.05):
    """Residual z-score: how far the actual content sits from the model's
    expectation, in units of how sure the model was. Needs x - which is exactly
    what v2's variance-only score was missing."""
    return np.abs(x - mu) / (sd + eps)

N_PASSES, STRIDE, POS_BATCH = 20, 12, 6      # PATCH is 16, set in section 3

@torch.no_grad()
def mc_uncertainty(x_np, n=N_PASSES, patch=PATCH, stride=STRIDE, pos_batch=POS_BATCH):
    """Sliding-window inpainting. Returns (mean fill, std across passes).
    Hole positions are batched so the finer stride stays affordable."""
    model.train()                                          # dropout ON
    x = torch.from_numpy(x_np)[None, None].to(DEV)
    acc_mu = torch.zeros(H, W, device=DEV)
    acc_sd = torch.zeros(H, W, device=DEV)
    cnt    = torch.zeros(H, W, device=DEV)

    positions = [(y, x0) for y in range(0, H - patch + 1, stride)
                         for x0 in range(0, W - patch + 1, stride)]

    for i in range(0, len(positions), pos_batch):
        chunk = positions[i:i+pos_batch]; k = len(chunk)
        holes = torch.zeros(k, 1, H, W, device=DEV)
        for j, (y, x0) in enumerate(chunk):
            holes[j, :, y:y+patch, x0:x0+patch] = 1.
        xin   = (x * (1 - holes)).repeat_interleave(n, 0)   # k*n, n draws each
        hh    = holes.repeat_interleave(n, 0)
        preds = model(xin, hh).view(k, n, H, W)
        for j, (y, x0) in enumerate(chunk):
            reg = preds[j, :, y:y+patch, x0:x0+patch]
            acc_mu[y:y+patch, x0:x0+patch] += reg.mean(0)
            acc_sd[y:y+patch, x0:x0+patch] += reg.std(0)
            cnt[y:y+patch, x0:x0+patch]    += 1

    cnt = cnt.clamp(min=1)
    return (acc_mu/cnt).cpu().numpy(), (acc_sd/cnt).cpu().numpy()

t0 = time.time()
x = load_bscan(N_TRAIN + 1)
mu, sd = mc_uncertainty(x)
z = anomaly_score(x, mu, sd)
mask = retina_mask(mu)
print(f"one scan in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(1, 5, figsize=(19, 2.8))
for a, (im, t, cm) in zip(ax, [(x,"input","gray"), (mu,"model's expectation","gray"),
                               (sd,"pass disagreement (v2 used this)","inferno"),
                               (z,"residual z (v3 score)","inferno"),
                               (mask,"retina mask (ILM-RPE)","gray")]):
    a.imshow(im, cmap=cm); a.set_title(t, fontsize=8); a.axis("off")
plt.show()
print(f"mean residual z over retina: {z[mask>0].mean():.4f}")

## 6. The decision rule

Unchanged from v1 — this part was never the problem. Three statistics on the uncertainty
field, `u_total` gating first so that calm scans never reach the shape test.

| Statistic | Meaning | High when |
|---|---|---|
| `u_total` | mean uncertainty over the retina | unsure overall |
| `top_mass` | share of uncertainty in the hottest 5% of pixels | **concentrated** |
| `blob_share` | share of supra-threshold mass in the largest connected component | that concentration is **one region** |

In [ ]:
def uncertainty_stats(u, mask=None, top_frac=0.05, thresh_pct=80):
    u = np.asarray(u, dtype=np.float64)
    if mask is not None:
        vals = u[mask > 0]; masked = np.where(mask > 0, u, 0.)
    else:
        vals = u.ravel(); masked = u
    if vals.size == 0 or vals.sum() <= 0:
        return dict(u_total=0., top_mass=0., blob_share=0.)
    u_total = float(vals.mean())
    k = max(1, int(round(top_frac * vals.size)))
    top_mass = float(np.sort(vals)[-k:].sum() / vals.sum())
    binary = masked > np.percentile(vals, thresh_pct)
    if not binary.any():
        blob_share = 0.
    else:
        lab, n = ndimage.label(binary)
        m = ndimage.sum(masked, lab, index=np.arange(1, n+1))
        blob_share = float(m.max()/m.sum()) if n else 0.
    return dict(u_total=u_total, top_mass=top_mass, blob_share=blob_share)

def classify_shape(s, t_low, t_top, t_blob):
    if s["u_total"] < t_low: return "low", "cleared"
    conc = (s["top_mass"] >= t_top) and (s["blob_share"] >= t_blob)
    return ("focal", "review") if conc else ("diffuse", "rescan")

REASONS = {
 "cleared": "The model agrees with itself across all passes.",
 "rescan":  "Uncertainty is diffuse \u2014 this is a capture problem, not a clinical one. "
            "Likely small pupil or poor fixation.",
 "review":  "The capture is fine; the anatomy is genuinely ambiguous. A rescan won't help.",
}

## 7. The experiment that can fail

### The degradation model, rewritten

v1 led with Gaussian blur, which for a reconstruction model makes the input *easier* to
predict — degraded scans scored calmer than clean ones. Real OCT capture failure is not
blur. It is signal attenuation from a small pupil or media opacity, multiplicative speckle
at low signal, shadowing from vessels and floaters, tilt from poor centration, and motion
breaks. None of those make an image easier to reconstruct.

### Read the diagnostic panel first

Before the histograms, look at the anomaly diagnostic. If the injected lesion appears
cleanly in the model's fill, the model is reproducing pathology it has never seen and the
whole approach is dead in the water — which is precisely what happened in v1.

In [ ]:
def degrade(x, seed):
    r = np.random.default_rng(seed)
    y = x.astype(np.float32).copy(); Hh, Ww = y.shape
    y = y * r.uniform(.45, .70) + r.uniform(.02, .08)          # attenuation
    y = y * (1. + r.normal(0, r.uniform(.25, .45), y.shape))   # multiplicative speckle
    for _ in range(r.integers(2, 6)):                          # vessel / floater shadows
        c = r.integers(0, Ww); w = r.integers(3, 12)
        y[:, max(0, c-w):c+w] *= r.uniform(.25, .55)
    shift = np.linspace(-r.uniform(4,14), r.uniform(4,14), Ww).astype(int)   # tilt
    y = np.stack([np.roll(y[:,c], shift[c]) for c in range(Ww)], axis=1)
    if r.random() < .6:                                        # motion breaks
        for _ in range(r.integers(2, 5)):
            rr = r.integers(0, Hh-3)
            y[rr:rr+3] = np.roll(y[rr:rr+3], r.integers(-10, 10), axis=1)
    return np.clip(y, 0, 1).astype(np.float32)

def inject_anomaly(x, seed):
    r = np.random.default_rng(seed); y = x.copy(); Hh, Ww = y.shape
    cy, cx = int(r.uniform(.45,.72)*Hh), int(r.uniform(.25,.75)*Ww)
    ry, rx = int(r.uniform(5,10)), int(r.uniform(14,26))
    yy, xx = np.mgrid[0:Hh, 0:Ww]
    m = (((xx-cx)/rx)**2 + ((yy-cy)/ry)**2) < 1
    y[m] *= r.uniform(.10, .30)
    # v2 BUG: gaussian_filter was applied to the WHOLE image, smoothing speckle
    # everywhere and dragging the anomalous group's residual down. Local only.
    soft = ndimage.gaussian_filter(y, .6)
    edge = ndimage.binary_dilation(m, np.ones((5,5)))
    y[edge] = soft[edge]
    return np.clip(y, 0, 1).astype(np.float32)

In [ ]:
# --- diagnostic: read this BEFORE the histograms ---
x0 = load_bscan(N_TRAIN + 3)
xa = inject_anomaly(x0, 3)
mu_a, sd_a = mc_uncertainty(xa)
z_a = anomaly_score(xa, mu_a, sd_a)

fig, ax = plt.subplots(1, 5, figsize=(19, 2.8))
for a, (im, t, cm) in zip(ax, [(x0,"normal","gray"), (xa,"anomaly injected","gray"),
                               (mu_a,"model's expectation","gray"),
                               (np.abs(xa-mu_a),"raw residual","inferno"),
                               (z_a,"residual z (the score)","inferno")]):
    a.imshow(im, cmap=cm); a.set_title(t, fontsize=8); a.axis("off")
plt.suptitle("The lesion should be ABSENT from the expectation and BRIGHT in the last two.")
plt.show()

In [ ]:
N_EVAL = 25                      # finer stride is slower per scan

def stats_for(fn, idx):
    out = []
    for i in idx:
        x0 = load_bscan(i)
        xx = x0 if fn is None else fn(x0, i)
        mu, sd = mc_uncertainty(xx)
        z = anomaly_score(xx, mu, sd)                  # <-- the v3 change
        out.append(uncertainty_stats(z, retina_mask(mu)))
    return out

IDX = list(range(N_TRAIN, N_TRAIN + N_EVAL))
t0 = time.time()
s_clean = stats_for(None, IDX)
s_degra = stats_for(degrade, IDX)
s_anom  = stats_for(inject_anomaly, IDX)
print(f"{3*N_EVAL} scans in {time.time()-t0:.0f}s\n")

for name, ss in [("clean", s_clean), ("degraded", s_degra), ("anomalous", s_anom)]:
    for k in ("u_total", "top_mass", "blob_share"):
        v = np.array([s[k] for s in ss])
        print(f"  {name:10s} {k:11s} mean={v.mean():.4f}  p10={np.quantile(v,.1):.4f}  p90={np.quantile(v,.9):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for a, k in zip(ax, ["u_total", "top_mass", "blob_share"]):
    for ss, lab, c in [(s_clean,"clean","#1A5FC8"), (s_degra,"degraded","#E0A529"),
                       (s_anom,"anomalous","#C25200")]:
        a.hist([s[k] for s in ss], bins=14, alpha=.6, label=lab, color=c)
    a.set_title(k); a.legend(fontsize=8)
plt.suptitle("Separation is the thesis. Overlap = the three-outcome model collapses to two.")
plt.tight_layout(); plt.show()

# quick numeric read: effect size between groups
def cohen_d(a, b):
    a, b = np.array(a), np.array(b)
    s = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / max(s, 1e-9)

print("Effect sizes (Cohen's d, |d|>0.8 is a large effect):")
for k in ("u_total", "top_mass", "blob_share"):
    d1 = cohen_d([s[k] for s in s_degra], [s[k] for s in s_clean])
    d2 = cohen_d([s[k] for s in s_anom],  [s[k] for s in s_clean])
    d3 = cohen_d([s[k] for s in s_anom],  [s[k] for s in s_degra])
    print(f"  {k:11s} degraded-vs-clean {d1:+.2f}   anomalous-vs-clean {d2:+.2f}   "
          f"anomalous-vs-degraded {d3:+.2f}")

## 8. Thresholds, confusion, rescan resolution

`CLEAR_RATE` is the safety dial. Raising it clears more scans and raises your headline
"scans you didn't have to open" number, at the cost of more pathology sitting in the
cleared bucket. Set it with a clinician, not by what makes the demo look best.

In [ ]:
CLEAR_RATE = 0.85

t_low = float(np.quantile([s["u_total"] for s in s_clean], CLEAR_RATE))
dt = np.array([s["top_mass"] for s in s_degra]); db = np.array([s["blob_share"] for s in s_degra])
at = np.array([s["top_mass"] for s in s_anom]);  ab = np.array([s["blob_share"] for s in s_anom])
TH = dict(t_low=t_low,
          t_top=float((np.quantile(dt,.9)+np.quantile(at,.1))/2),
          t_blob=float((np.quantile(db,.9)+np.quantile(ab,.1))/2))
print("thresholds:", {k: round(v,5) for k,v in TH.items()})

print("\nConfusion (rows = truth, cols = decision):")
hdr = f"{'':11s} {'cleared':>8s} {'rescan':>8s} {'review':>8s}   correct"
print(hdr); print("-"*len(hdr))
exp = {"clean":"cleared","degraded":"rescan","anomalous":"review"}
for nm, ss in [("clean",s_clean),("degraded",s_degra),("anomalous",s_anom)]:
    p = [classify_shape(s, **TH)[1] for s in ss]
    c = {d: p.count(d) for d in ("cleared","rescan","review")}
    print(f"{nm:11s} {c['cleared']:8d} {c['rescan']:8d} {c['review']:8d}   "
          f"{p.count(exp[nm])/len(p):.0%}")
json.dump(TH, open("thresholds.json","w"))

In [ ]:
flagged = [i for i, s in zip(IDX, s_degra) if classify_shape(s, **TH)[1] == "rescan"]
resolved = 0
for i in flagged:
    xr = load_bscan(i)                                          # the "retake"
    mu, sd = mc_uncertainty(xr)
    if classify_shape(uncertainty_stats(anomaly_score(xr, mu, sd),
                                        retina_mask(mu)), **TH)[1] == "cleared":
        resolved += 1
rate = resolved / max(len(flagged), 1)
print(f"rescan resolution rate: {resolved}/{len(flagged)} = {rate:.0%}")
print("PASS \u2014 diffuse uncertainty tracks capture quality" if rate >= .5
      else "FAIL \u2014 rescan flag is not a capture signal.")

In [ ]:
from sklearn.isotonic import IsotonicRegression
raw, corr = [], []
for ss, e in [(s_clean,"cleared"), (s_degra,"rescan"), (s_anom,"review")]:
    for s in ss:
        raw.append(float(np.clip(100*np.exp(-(s["u_total"]/max(t_low,1e-9))/3), 1, 99)))
        corr.append(1. if classify_shape(s, **TH)[1] == e else 0.)
raw, corr = np.array(raw), np.array(corr)
iso = IsotonicRegression(y_min=.01, y_max=.99, out_of_bounds="clip").fit(raw, corr)

def ece(c, y, bins=10):
    e, n = 0., len(c)
    for lo, hi in zip(np.linspace(0,1,bins+1)[:-1], np.linspace(0,1,bins+1)[1:]):
        m = (c > lo) & (c <= hi)
        if m.sum(): e += m.sum()/n * abs(y[m].mean() - c[m].mean())
    return e

print(f"ECE before calibration: {ece(raw/100, corr):.3f}")
print(f"ECE after  calibration: {ece(iso.predict(raw), corr):.3f}")
import pickle; pickle.dump(iso, open("calibrator.pkl","wb"))

## 9. `analyze()` — the TriageResult contract

Keys must match the `TriageResult` interface in `coherence.jsx` exactly or the UI renders
blanks.

In [ ]:
def to_data_url(arr, cmap=None):
    a = np.asarray(arr, dtype=np.float64)
    a = (a - a.min()) / max(np.ptp(a), 1e-9)
    img = (Image.fromarray((plt.get_cmap(cmap)(a)*255).astype(np.uint8), "RGBA") if cmap
           else Image.fromarray((a*255).astype(np.uint8), "L"))
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def analyze_array(x, case_id=None, laterality="OD"):
    mu, sd = mc_uncertainty(x)
    u = anomaly_score(x, mu, sd)
    mask = retina_mask(mu)
    s = uncertainty_stats(u, mask)
    shape, decision = classify_shape(s, **TH)
    rawc = float(np.clip(100*np.exp(-(s["u_total"]/max(TH["t_low"],1e-9))/3), 1, 99))
    conf = int(round(float(iso.predict([rawc])[0])*100))

    # PLACEHOLDER: tissue-row count x axial scale. NOT a measurement.
    # Replace with real layer segmentation + DICOM pixel spacing before clinical use.
    thickness = float(mask.sum(axis=0).mean() * 3.87)
    unc_um = float(np.clip(u[mask>0].mean()*120, 3, 40))

    return {
        "caseId": case_id or str(uuid.uuid4())[:8].upper(),
        "confidence": conf,
        "uncertaintyShape": shape,
        "decision": decision,
        "reason": REASONS[decision],
        "heatmapUrl": to_data_url(u, cmap="inferno"),
        "scanUrl": to_data_url(x),
        "layerThicknessUm": int(round(thickness)),
        "measurementUncertaintyUm": int(round(unc_um)),
        "laterality": laterality,
        "prior": None,
        "_stats": {k: round(v,5) for k,v in s.items()},
    }

demo = analyze_array(degrade(load_bscan(N_TRAIN+5), 5))
print(json.dumps({k:v for k,v in demo.items() if not str(v).startswith("data:")}, indent=2))

## 10. Serve it

FastAPI + ngrok. Put the printed URL in `.env.local` as `VITE_ENGINE_URL`, set
`VITE_ENGINE=real`, restart the Vite dev server.

Inference is a sliding window now, so expect a few seconds per scan rather than
milliseconds. Fine for a demo; something to optimise before a pilot.

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio, uvicorn, pydicom
from pyngrok import ngrok

app = FastAPI(title="Coherence RealEngine v2")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

def bytes_to_bscan(raw, name=""):
    if name.lower().endswith((".dcm",".dicom")) or raw[128:132] == b"DICM":
        ds = pydicom.dcmread(io.BytesIO(raw), force=True)
        arr = ds.pixel_array
        if arr.ndim == 3: arr = arr[arr.shape[0]//2]
        lat = str(getattr(ds,"Laterality","") or getattr(ds,"ImageLaterality","") or "OD")
    else:
        arr, lat = np.asarray(Image.open(io.BytesIO(raw)).convert("L")), "OD"
    arr = np.asarray(Image.fromarray(arr.astype(np.float32)).resize((W, H)))
    arr = (arr - arr.min()) / max(np.ptp(arr), 1e-9)
    return arr.astype(np.float32), ("OS" if lat.upper().startswith("L") else "OD")

@app.get("/health")
def health():
    return {"ok": True, "engine": "real-v3-residual", "passes": N_PASSES,
            "data": "kermany" if DATA_OK else "SYNTHETIC", "thresholds": TH}

@app.post("/api/analyze")
async def api_analyze(scan: UploadFile = File(...)):
    raw = await scan.read()
    x, lat = bytes_to_bscan(raw, scan.filename or "")
    return analyze_array(x, laterality=lat)

# ngrok.set_auth_token("YOUR_TOKEN")
url = ngrok.connect(8000).public_url
print("\n" + "="*62)
print(f"  VITE_ENGINE_URL={url}")
print(f"  VITE_ENGINE=real")
print("="*62 + "\n")
nest_asyncio.apply(); uvicorn.run(app, port=8000)

## 11. How to read the result

**Check in this order.** Each one can kill the run, so stop at the first failure rather
than reading further.

1. **Section 4 sanity check.** Do the layers continue plausibly across the hole? If the
   fill is grey mush, training failed and nothing downstream means anything.
2. **Section 7 diagnostic panel.** The lesion should be *absent* from the model's
   expectation (it is inpainted away, which is correct) and *bright* in the raw residual
   and the z-score. If the residual is bright everywhere instead of at the lesion, the
   expectation is too blurry to serve as a reference — go to `PATCH=8, STRIDE=6`.
3. **Effect sizes.** `anomalous-vs-clean` on `top_mass` and `blob_share` is the number
   that matters most. Below about 0.5, the focal signal is too weak to build a product on.
4. **Confusion matrix and rescan resolution rate.**

### If it separates

Good, and hold it lightly. A synthetic dark ellipse is not drusen. The published failure
mode of this whole family of methods is missing **small** lesions, because the network
keeps segmenting confidently around them — and early AMD is exactly the population an
optometrist most needs help with. Next step is labelled pathology, not a pilot.

### If it does not separate

That is a real result about the thesis, arrived at on a free GPU instead of in month eight
of a pilot. Before concluding, rule out the cheap explanations in this order:

- **Patch still too large.** Try `PATCH=8, STRIDE=6`. Smaller holes make the expectation
  more local and sharper, at roughly 3x the inference time.
- **Not enough training.** Inpainting is harder than reconstruction. If val loss was still
  falling at epoch 18, raise `EPOCHS`.
- **Scan-quality heterogeneity swamping the signal.** In v1, clean `u_total` ranged
  0.09-0.23 across scans that were all supposedly normal. If that spread persists,
  per-scan quality is dominating anatomy and needs standardising before the statistics can
  see anything.
- **MC dropout too weak an estimator.** The literature consistently finds deep ensembles
  calibrate better for selective referral. Five models at different seeds, mean and
  variance across them instead of across dropout draws.
- **The denominator is doing harm.** If `sd` is near-uniform, `z` is just a rescaled
  residual and the division adds noise. Compare against plain `|x - mu|` — one line,
  and worth knowing which term carries the signal.

If none of those rescue it, the honest conclusion is that uncertainty shape does not
separate capture failure from pathology in OCT at this scale — and the three-outcome model
becomes two: cleared and review. That is still a product. It is just a different one, and
better to know now.